## 얼굴 표정으로 감정 분석(MediaPipe) + 감정에 따른 AI 응답 웹앱

In [4]:
# ! pip install mediapipe opencv-python numpy

In [2]:
from flask import Flask, render_template, Response, request, jsonify
import cv2
from emotion_detector import detect_emotion, get_current_emotion
from openai import OpenAI
import os
from dotenv import load_dotenv

# .env 파일의 내용 불러오기
load_dotenv("C:/env/.env")

# 환경 변수 가져오기
API_KEY = os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key=API_KEY)

app = Flask(__name__)

camera = cv2.VideoCapture(0)
# camera = cv2.VideoCapture(1)

def gen_frames():
    while True:
        success, frame = camera.read()
        if not success:
            break
        else:
            detect_emotion(frame)
            ret, buffer = cv2.imencode('.jpg', frame)
            frame = buffer.tobytes()
            yield (b'--frame\r\n'
                   b'Content-Type: image/jpeg\r\n\r\n' + frame + b'\r\n')

@app.route('/')
def index():
    return render_template('index.html')

@app.route('/video_feed')
def video_feed():
    return Response(gen_frames(),
                    mimetype='multipart/x-mixed-replace; boundary=frame')

@app.route('/current_emotion')
def current_emotion_api():
    return jsonify({"emotion": get_current_emotion()})

@app.route('/get_response', methods=['POST'])
def get_response():
    emotion = request.json['emotion']
    print(emotion)
    
    prompt = f"사용자가 {emotion}한 감정 상태이다. 이 감정에 맞게 공감하는 대화를 해줘."

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "너는 감정을 이해하고 공감하는 따뜻한 대화 전문가야."},
            {"role": "user", "content": prompt}
        ]
    )
    ai_message = response.choices[0].message.content
    return jsonify({"response": ai_message})

if __name__ == "__main__":
    app.run(debug=False)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [21/May/2026 17:15:32] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [21/May/2026 17:15:32] "GET /video_feed HTTP/1.1" 200 -
127.0.0.1 - - [21/May/2026 17:15:37] "GET /current_emotion HTTP/1.1" 200 -


happy


127.0.0.1 - - [21/May/2026 17:15:40] "POST /get_response HTTP/1.1" 200 -


happy


127.0.0.1 - - [21/May/2026 17:15:42] "POST /get_response HTTP/1.1" 200 -
127.0.0.1 - - [21/May/2026 17:15:42] "GET /current_emotion HTTP/1.1" 200 -


In [3]:
# import sys
# !{sys.executable} -m pip uninstall tensorflow tensorflow-intel -y
# !{sys.executable} -m pip install mediapipe==0.10.9 "protobuf>=3.20.3,<5"